In [14]:
import asyncio
import datetime
import logging
import os
from typing import Dict, List, Union
from providers.completion import PROVIDER_CLASSES
import nest_asyncio

import yaml
from models.llm import CompletionsModel
from sqlalchemy import select
from sqlalchemy.ext.asyncio import AsyncSession, create_async_engine
from sqlalchemy.orm import sessionmaker, Session
from sqlalchemy import create_engine, select, func, desc, asc, case


from orchestra.db.models.orchestra_models import (  # noqa: WPS235
    BenchmarkRun,
    Datapoint,
    Endpoint,
    Model,
    Provider,
)
nest_asyncio.apply()

def create_db_session() -> sessionmaker:  # noqa: WPS210
    # noqa: DAR201
    """Creates an async db session.

    If ORCHESTRA_<> env vars are not defined, it defaults to
    orchestra:orchestra@localhost:5432/orchestra.

    Returns:
        sessionmaker: Async SQLAlchemy session maker.
    """
    user = os.getenv("ORCHESTRA_DB_USER", "orchestra")
    password = os.getenv("ORCHESTRA_DB_PASS", "orchestra")
    host = os.getenv("ORCHESTRA_DB_HOST", "localhost")
    port = os.getenv("ORCHESTRA_DB_PORT", "5432")
    db_name = os.getenv("ORCHESTRA_DB_BASE", "orchestra")
    db_url = f"postgresql+asyncpg://{user}:{password}@{host}:{port}/{db_name}"  # noqa: WPS221, E501
    # TODO: logger.info(db_url)
    engine = create_async_engine(db_url)
    return Session(engine, expire_on_commit=False)



In [18]:
nest_asyncio.apply()

session = create_db_session()

In [19]:
nest_asyncio.apply()

async def run_query():
    # Create a SQLAlchemy session
    db_session = create_db_session()

    # Step 1: Select the latest metrics for each configuration for each provider
    latest_metrics_query = (
        db_session.query(
            Provider.name.label('provider'),
            func.concat(
                BenchmarkRun.regime, '_', BenchmarkRun.region, '_', BenchmarkRun.seq_len
            ).label('config'),
            Datapoint.metric_name.label('metric'),
            func.array_agg(Datapoint.value, order_by=desc(Datapoint.measured_at)).label('metric_values')
        )
        .join(Endpoint, BenchmarkRun.endpoint_id == Endpoint.id)
        .join(Model, Endpoint.mdl_id == Model.id)
        .join(Provider, Endpoint.provider_id == Provider.id)
        .filter(Model.mdl_code.in_(['llama-2-70b-chat', 'mixtral-8x7b']))
        .group_by('provider', 'config', 'metric')
    )

    # Print results for Step 1
    print("Step 1 Results:")
    async for row in latest_metrics_query.all():
        print(row)

    # Continue with the rest of the steps...
    # Make sure to adjust the model classes and add necessary steps for each part of your query

    # Close the SQLAlchemy session
    await db_session.close()

In [20]:
nest_asyncio.apply()

await run_query()

Step 1 Results:


AsyncContextNotStarted: AsyncConnection context has not been started and object has not been awaited.